In [1]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

# Date range — aligns with EPS surprises window
FROM_DATE = "2023-01-01"
TO_DATE = "2025-12-31"

data = finnhub_client.stock_lobbying(symbol="AAPL", _from=FROM_DATE, to=TO_DATE)

In [2]:
# Top level keys
print("=== TOP LEVEL KEYS ===")
for key in data.keys():
    print(f"  {key}")

# How many lobbying activities returned
print(f"\nLobbying activities returned: {len(data['data'])}")

=== TOP LEVEL KEYS ===
  data
  symbol

Lobbying activities returned: 102


In [3]:
# Look at first activity in full
print("=== FIRST ACTIVITY ===")
for k, v in data["data"][0].items():
    print(f"  {k}: {v}")

=== FIRST ACTIVITY ===
  symbol: AAPL
  name: APPLE, INC.
  description: Technology company
  country: US
  year: 2024
  period: Q4
  documentUrl: https://lda.senate.gov/filings/public/filing/2c6c1a3d-546a-4e2e-b7cc-53f4eb432c97/print/
  income: 40000
  expenses: None
  postedName: 
  date: 
  clientId: 63242
  registrantId: 401109033
  senateId: 401109033-63242
  houseRegistrantId: 


In [4]:
# Understand the range of expenses and dates
print("=== EXPENSES ACROSS ALL ACTIVITIES ===")
for activity in data["data"]:
    print(f"  {activity['date']} | {activity['period']} | expenses: {activity['expenses']} | income: {activity['income']}")

=== EXPENSES ACROSS ALL ACTIVITIES ===
   | Q4 | expenses: None | income: 40000
   | Q1 | expenses: None | income: 60000
   | Q2 | expenses: None | income: 60000
   | Q3 | expenses: None | income: 60000
   | Q3 | expenses: None | income: 50000
   | Q3 | expenses: None | income: 20000
   | Q4 | expenses: 2870000 | income: None
   | Q1 | expenses: 2660000 | income: None
   | Q2 | expenses: 2410000 | income: None
   | Q3 | expenses: 1840000 | income: None
   | Q4 | expenses: 2950000 | income: None
   | Q1 | expenses: 2130000 | income: None
   | Q2 | expenses: 2130000 | income: None
   | Q3 | expenses: 1900000 | income: None
   | Q4 | expenses: 1660000 | income: None
   | Q1 | expenses: 2450000 | income: None
   | Q2 | expenses: 2290000 | income: None
   | Q3 | expenses: 2530000 | income: None
   | Q4 | expenses: None | income: 90000
   | Q1 | expenses: None | income: 90000
   | Q2 | expenses: None | income: 90000
   | Q3 | expenses: None | income: 90000
   | Q4 | expenses: None | income: 

### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [ ]:
import finnhub
import os
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv()

finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

FROM_DATE = "2023-01-01"
TO_DATE   = "2025-12-31"

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
errors  = []

# ── field-level tracking ───────────────────────────────────────────────────
# Top-level wrapper fields (symbol, data)
wrapper_present = defaultdict(int)
wrapper_null    = defaultdict(int)
wrapper_types   = defaultdict(set)

# Activity-level fields (inside data[])
activity_present = defaultdict(int)
activity_null    = defaultdict(int)
activity_types   = defaultdict(set)

# ── activity count tracking ────────────────────────────────────────────────
activity_counts      = []   # activities per ticker that HAS lobbying
no_activity_tickers  = []   # tickers with valid response but empty data[]
total_activities     = 0

total = len(tickers)

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.stock_lobbying(symbol=ticker, _from=FROM_DATE, to=TO_DATE)

        if not data:
            errors.append((ticker, "empty response — no wrapper returned"))
            results[ticker] = {}
        else:
            results[ticker] = data

            # ── wrapper-level fields ──────────────────────────────────────
            for field, value in data.items():
                if field == "data":
                    continue  # handled separately below
                if value is None or value == "":
                    wrapper_null[field] += 1
                else:
                    wrapper_present[field] += 1
                    wrapper_types[field].add(type(value).__name__)

            # ── activity-level fields ─────────────────────────────────────
            activities = data.get("data", [])
            if not activities:
                no_activity_tickers.append(ticker)
            else:
                activity_counts.append(len(activities))
                total_activities += len(activities)

                for activity in activities:
                    for field, value in activity.items():
                        if value is None or value == "":
                            activity_null[field] += 1
                        else:
                            activity_present[field] += 1
                            activity_types[field].add(type(value).__name__)

    except Exception as e:
        errors.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(2)

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
successful    = len([r for r in results.values() if r])
empty_tickers = [t for t, r in results.items() if not r]
with_activity = len(activity_counts)

print(f"✅ Successfully pulled:              {successful} / {total}")
print(f"❌ Errors:                           {len(errors)}")
print(f"📭 Empty responses:                  {empty_tickers if empty_tickers else 'None'}")
print(f"📋 Tickers with lobbying activity:   {with_activity} / {total}")
print(f"⭕ Tickers with no activity:         {len(no_activity_tickers)} / {total}")
if no_activity_tickers:
    print(f"   {no_activity_tickers}")
print()

if errors:
    print("─── Errors ───")
    for ticker, msg in errors:
        print(f"   {ticker}: {msg}")
    print()

# ─────────────────────────────────────────
# SECTION 1 — ACTIVITY COUNT DISTRIBUTION
# ─────────────────────────────────────────
if activity_counts:
    print("─── Activity Count Distribution (tickers with lobbying only) ───")
    print(f"  Total activities across all tickers: {total_activities}")
    print(f"  Min activities per ticker:           {min(activity_counts)}")
    print(f"  Max activities per ticker:           {max(activity_counts)}")
    print(f"  Avg activities per ticker:           {sum(activity_counts) / len(activity_counts):.1f}")
    print()

# ─────────────────────────────────────────
# SECTION 2 — WRAPPER FIELD DECISION TABLE
# Fields at the top level of the response (e.g. symbol)
# ─────────────────────────────────────────
print("─── Wrapper Field Decision Table ───")
print(f"{'Field':<20} {'Present':>12} {'Null':>8} {'Types':<20} Recommendation")
print("─" * 85)

all_wrapper_fields = set(wrapper_present.keys()) | set(wrapper_null.keys())
for field in sorted(all_wrapper_fields):
    present    = wrapper_present.get(field, 0)
    null_count = wrapper_null.get(field, 0)
    types      = ", ".join(wrapper_types.get(field, {"unknown"}))
    rec        = "Optional" if null_count > 0 else "Required"
    print(f"{field:<20} {present:>9} seen  {null_count:>5} null   {types:<20} {rec}")

print()

# ─────────────────────────────────────────
# SECTION 3 — ACTIVITY FIELD DECISION TABLE
# Fields inside each lobbying activity record
# ─────────────────────────────────────────
print("─── Activity Field Decision Table (per lobbying record) ───")
print(f"{'Field':<30} {'Present':>12} {'Null':>8} {'Types':<20} Recommendation")
print("─" * 95)

all_activity_fields = set(activity_present.keys()) | set(activity_null.keys())
for field in sorted(all_activity_fields):
    present    = activity_present.get(field, 0)
    null_count = activity_null.get(field, 0)
    types      = ", ".join(activity_types.get(field, {"unknown"}))
    rec        = "Optional  ← null in some activities" if null_count > 0 else "Required  ← never null"
    print(f"{field:<30} {present:>9} seen  {null_count:>5} null   {types:<20} {rec}")

✅ Successfully pulled:              60 / 60
❌ Errors:                           0
📭 Empty responses:                  None
📋 Tickers with lobbying activity:   46 / 60
⭕ Tickers with no activity:         14 / 60
   ['BAESY', 'HEI', 'DRS', 'DCO', 'BP', 'SLB', 'EOG', 'FANG', 'CTRA', 'AR', 'CHRD', 'MTDR', 'IBM', 'DDOG']

─── Activity Count Distribution (tickers with lobbying only) ───
  Total activities across all tickers: 3012
  Min activities per ticker:           1
  Max activities per ticker:           272
  Avg activities per ticker:           65.5

─── Wrapper Field Decision Table ───
Field                     Present     Null Types                Recommendation
─────────────────────────────────────────────────────────────────────────────────────
symbol                      60 seen      0 null   str                  Required

─── Activity Field Decision Table (per lobbying record) ───
Field                               Present     Null Types                Recommendation
───────────